## 0. 加载 Qwen 模型

本 Notebook 使用 **Qwen2.5-7B-Instruct**（通过 ModelScope 加载）作为真实 LLM 后端，替代原有的模拟 LLM。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

### 安装依赖（首次运行需要）

```bash
pip install modelscope torch transformers
```


In [ ]:
# ============================================================
# 加载 Qwen 模型（通过 ModelScope）
# ============================================================
# 如果没有 GPU 或显存不足，可将模型 ID 改为 Qwen/Qwen2.5-3B-Instruct
# ============================================================

import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer


class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装类"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测 GPU / CPU
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("[QwenLLM] 模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """调用模型进行对话"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """重置对话历史"""
        self.messages = []


# 初始化 QwenLLM 实例
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
print("\n模型就绪，可以开始使用了。")

# 02 - 高级工具使用与 Function Calling

## 学习目标

- 理解 Function Calling 的底层原理
- 掌握工具定义、注册和调用的完整流程
- 学习工具链（Tool Chaining）和工具选择策略
- 实现一个支持多工具的智能 Agent

---

## 1. Function Calling 深度解析

### 1.1 什么是 Function Calling？

**Function Calling** 是 LLM 的一种能力，允许模型：

1. **识别** 何时需要调用外部工具
2. **生成** 正确的函数调用参数
3. **接收** 工具执行结果并继续推理

### 1.2 Function Calling 流程

```
┌─────────────────────────────────────────────────────────────┐
│                  Function Calling 流程                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  用户提问: "北京今天天气怎么样?"                            │
│                                                             │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────┐                                               │
│  │  LLM    │                                               │
│  │ 推理    │                                               │
│  └────┬────┘                                               │
│       │                                                     │
│       ▼                                                     │
│  识别需要调用 get_weather 工具                               │
│       │                                                     │
│       ▼                                                     │
│  生成函数调用:                                               │
│  {                                                          │
│    "name": "get_weather",                                  │
│    "arguments": {"city": "北京"}                          │
│  }                                                          │
│       │                                                     │
│       ▼                                                     │
│  执行工具 ──► 返回结果: {"temp": 25, "weather": "晴"}      │
│       │                                                     │
│       ▼                                                     │
│  LLM 基于结果生成最终回答:                                   │
│  "北京今天天气晴朗，气温25度。"                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 1.3 工具定义格式（OpenAI 格式）

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称",
                    },
                    "date": {
                        "type": "string",
                        "description": "日期，格式: YYYY-MM-DD"
                    }
                },
                "required": ["city"]
            }
        }
    }
]
```

---

## 2. 工具系统架构

### 2.1 工具注册与发现

```
┌─────────────────────────────────────────────┐
│              工具注册中心                     │
├─────────────────────────────────────────────┤
│                                             │
│  ┌─────────────┐    ┌──────────────────┐   │
│  │  工具定义    │───►│  工具注册表       │   │
│  │  (Schema)   │    │  (Tool Registry) │   │
│  └─────────────┘    └────────┬─────────┘   │
│                              │              │
│                              ▼              │
│  ┌─────────────┐    ┌──────────────────┐   │
│  │  LLM 请求   │◄───│  工具发现/选择    │   │
│  │  (带工具列表)│    │  (Tool Discovery)│   │
│  └─────────────┘    └──────────────────┘   │
│                                             │
└─────────────────────────────────────────────┘
```

### 2.2 工具执行流程

```
  LLM 输出函数调用
       │
       ▼
  解析函数名和参数
       │
       ▼
  查找对应的工具实现
       │
       ▼
  验证参数（类型检查）
       │
       ▼
  执行工具函数
       │
       ▼
  处理结果（成功/异常）
       │
       ▼
  返回结果给 LLM
```

---

## 3. 动手实现：工具系统

### 3.1 基础工具定义

In [ ]:
from typing import Dict, List, Callable, Any
import json
import random
from datetime import datetime

class Tool:
    """工具基类"""
    
    def __init__(self, name: str, description: str, parameters: Dict):
        self.name = name
        self.description = description
        self.parameters = parameters
        self.func: Callable = None
    
    def set_function(self, func: Callable):
        """设置工具函数"""
        self.func = func
        return self
    
    def execute(self, **kwargs) -> Any:
        """执行工具"""
        if self.func is None:
            raise ValueError(f"工具 {self.name} 未设置函数")
        
        # 验证必需参数
        required = self.parameters.get("required", [])
        for param in required:
            if param not in kwargs:
                raise ValueError(f"缺少必需参数: {param}")
        
        return self.func(**kwargs)
    
    def to_schema(self) -> Dict:
        """转换为 OpenAI 工具格式"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    
    def __repr__(self):
        return f"Tool({self.name}: {self.description[:30]}...)"

# 创建工具实例
weather_tool = Tool(
    name="get_weather",
    description="获取指定城市的天气信息",
    parameters={
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "城市名称，如'北京'、'上海'"
            },
            "date": {
                "type": "string",
                "description": "日期，格式: YYYY-MM-DD，默认为今天"
            }
        },
        "required": ["city"]
    }
)

# 定义工具函数
def get_weather_impl(city: str, date: str = None) -> Dict:
    """模拟获取天气"""
    weather_types = ["晴", "多云", "阴", "小雨", "大雨"]
    return {
        "city": city,
        "date": date or datetime.now().strftime("%Y-%m-%d"),
        "temperature": random.randint(15, 35),
        "weather": random.choice(weather_types),
        "humidity": random.randint(30, 90)
    }

weather_tool.set_function(get_weather_impl)

print("工具定义:")
print(json.dumps(weather_tool.to_schema(), indent=2, ensure_ascii=False))

print("\n工具执行:")
result = weather_tool.execute(city="北京")
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
class ToolRegistry:
    """工具注册中心"""
    
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool):
        """注册工具"""
        self.tools[tool.name] = tool
        print(f"✅ 注册工具: {tool.name}")
    
    def get_tool(self, name: str) -> Tool:
        """获取工具"""
        if name not in self.tools:
            raise ValueError(f"工具 '{name}' 未注册")
        return self.tools[name]
    
    def list_tools(self) -> List[str]:
        """列出所有工具"""
        return list(self.tools.keys())
    
    def get_schemas(self) -> List[Dict]:
        """获取所有工具的 Schema"""
        return [tool.to_schema() for tool in self.tools.values()]
    
    def execute(self, tool_name: str, **kwargs) -> Any:
        """执行工具"""
        tool = self.get_tool(tool_name)
        return tool.execute(**kwargs)

# 创建更多工具
search_tool = Tool(
    name="web_search",
    description="搜索网络信息",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "搜索关键词"
            },
            "num_results": {
                "type": "integer",
                "description": "返回结果数量",
                "default": 5
            }
        },
        "required": ["query"]
    }
).set_function(
    lambda query, num_results=5: [
        {"title": f"结果{i+1}: {query}", "url": f"https://example.com/{i}", "snippet": f"关于{query}的相关信息..."}
        for i in range(num_results)
    ]
)

calculator_tool = Tool(
    name="calculator",
    description="执行数学计算",
    parameters={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "数学表达式，如 '2 + 2' 或 'sqrt(16)'"
            }
        },
        "required": ["expression"]
    }
).set_function(
    lambda expression: {
        "expression": expression,
        "result": eval(expression),  # 注意：实际应用中需要安全的计算环境
        "timestamp": datetime.now().isoformat()
    }
)

# 注册所有工具
registry = ToolRegistry()
registry.register(weather_tool)
registry.register(search_tool)
registry.register(calculator_tool)

print(f"\n已注册工具: {registry.list_tools()}")
print("\n工具 Schemas:")
for schema in registry.get_schemas():
    print(f"  - {schema['function']['name']}")

### 3.2 模拟 LLM Function Calling

In [ ]:
# ---- 无模型时的备选方案：MockLLMWithTools（纯模拟）----
# class MockLLMWithTools:
#     def __init__(self, tool_registry):
#         self.tool_registry = tool_registry
#     def chat(self, query):
#         # ... 纯模拟逻辑 ...

class QwenLLMWithTools:
    """基于 QwenLLM 的工具调用 Agent
    
    注意：纯 transformers 推理不支持原生 Function Calling，
    因此工具选择仍使用关键词匹配逻辑，
    但回答生成部分使用 QwenLLM 真实模型调用。
    """
    
    def __init__(self, tool_registry, llm_model):
        self.tool_registry = tool_registry
        self.llm = llm_model  # QwenLLM 实例
        self.conversation_history = []
    
    def _should_use_tool(self, query: str) -> bool:
        """判断是否需要使用工具（关键词匹配）"""
        tool_keywords = {
            "get_weather": ["天气", "温度", "下雨", "晴"],
            "web_search": ["搜索", "查找", "查询", "什么是"],
            "calculator": ["计算", "等于", "+", "-", "*", "/", "sqrt"]
        }
        
        for tool_name, keywords in tool_keywords.items():
            if any(kw in query for kw in keywords):
                return True
        return False
    
    def _select_tool(self, query: str) -> Tuple[str, Dict]:
        """选择工具并生成参数（关键词匹配）"""
        if any(kw in query for kw in ["天气", "温度"]):
            cities = ["北京", "上海", "广州", "深圳", "杭州"]
            city = next((c for c in cities if c in query), "北京")
            return "get_weather", {"city": city}
        
        elif any(kw in query for kw in ["搜索", "查找"]):
            query_term = query.replace("搜索", "").replace("查找", "").strip()
            return "web_search", {"query": query_term or "AI Agent"}
        
        elif any(kw in query for kw in ["计算"]):
            expr = query.replace("计算", "").strip()
            return "calculator", {"expression": expr or "1+1"}
        
        return None, {}
    
    def chat(self, query: str) -> Dict:
        """聊天（支持工具调用 + QwenLLM 回答生成）"""
        print(f"\n用户: {query}")
        print("="*60)
        
        # 判断是否需要工具
        if not self._should_use_tool(query):
            # 不需要工具，直接使用 QwenLLM 回答
            response = self.llm.chat(query, max_new_tokens=256, temperature=0.7)
            result = {
                "query": query,
                "tool_calls": [],
                "response": response
            }
            self.conversation_history.append(result)
            return result
        
        # 选择工具（关键词匹配）
        tool_name, params = self._select_tool(query)
        
        if not tool_name:
            response = self.llm.chat(query, max_new_tokens=256, temperature=0.7)
            result = {
                "query": query,
                "tool_calls": [],
                "response": response
            }
            self.conversation_history.append(result)
            return result
        
        print(f"\nLLM 决定调用工具: {tool_name}")
        print(f"   参数: {json.dumps(params, ensure_ascii=False)}")
        
        # 执行工具
        try:
            tool_result = self.tool_registry.execute(tool_name, **params)
            print(f"\n工具返回结果:")
            print(f"   {json.dumps(tool_result, indent=2, ensure_ascii=False)[:200]}...")
            
            # 使用 QwenLLM 基于工具结果生成回答
            system_prompt = (
                "你是一个智能助手，请根据工具返回的结果，用简洁友好的中文回答用户的问题。"
            )
            user_msg = f"用户问题: {query}\n\n工具 {tool_name} 返回结果: {json.dumps(tool_result, ensure_ascii=False)}"
            response = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=256, temperature=0.7)
            
            result = {
                "query": query,
                "tool_calls": [{"tool": tool_name, "params": params, "result": tool_result}],
                "response": response
            }
            
        except Exception as e:
            result = {
                "query": query,
                "tool_calls": [{"tool": tool_name, "params": params, "error": str(e)}],
                "response": f"工具调用失败: {str(e)}"
            }
        
        self.conversation_history.append(result)
        return result

# 创建带工具的 LLM（使用 QwenLLM 实例）
llm_with_tools = QwenLLMWithTools(registry, llm)

# 测试工具调用
queries = [
    "北京今天天气怎么样?",
    "搜索一下 AI Agent 的最新进展",
    "计算 123 * 456",
    "你好，介绍一下自己"  # 不需要工具
]

for query in queries:
    result = llm_with_tools.chat(query)
    print(f"\n回答: {result['response']}")
    print("\n" + "="*60 + "\n")

### 3.3 工具链（Tool Chaining）

工具链允许一个工具的输出作为另一个工具的输入：

In [ ]:
class ToolChain:
    """工具链"""
    
    def __init__(self, tool_registry: ToolRegistry):
        self.tool_registry = tool_registry
        self.steps = []
    
    def add_step(self, tool_name: str, param_mapping: Dict[str, str]):
        """添加工具链步骤"""
        """
        param_mapping: 定义如何从上下文获取参数
        例如: {"city": "input.city", "date": "step1.result.date"}
        """
        self.steps.append({
            "tool": tool_name,
            "param_mapping": param_mapping
        })
        return self
    
    def execute(self, initial_input: Dict) -> List[Dict]:
        """执行工具链"""
        context = {"input": initial_input}
        results = []
        
        for i, step in enumerate(self.steps):
            step_name = f"step_{i+1}"
            print(f"\n🔗 执行步骤 {i+1}: {step['tool']}")
            
            # 解析参数
            params = self._resolve_params(step["param_mapping"], context)
            print(f"   参数: {params}")
            
            # 执行工具
            result = self.tool_registry.execute(step["tool"], **params)
            context[step_name] = {"result": result}
            results.append({
                "step": step_name,
                "tool": step["tool"],
                "params": params,
                "result": result
            })
            
            print(f"   结果: {json.dumps(result, indent=2, ensure_ascii=False)[:150]}...")
        
        return results
    
    def _resolve_params(self, mapping: Dict, context: Dict) -> Dict:
        """解析参数映射"""
        params = {}
        for key, path in mapping.items():
            value = self._get_value_from_path(context, path)
            params[key] = value
        return params
    
    def _get_value_from_path(self, obj: Dict, path: str) -> any:
        """从对象中按路径获取值"""
        parts = path.split(".")
        current = obj
        for part in parts:
            if isinstance(current, dict):
                current = current.get(part)
            else:
                return None
        return current

# 创建工具链：搜索 -> 计算
chain = ToolChain(registry)
chain.add_step("web_search", {"query": "input.topic"})

print("执行工具链: 搜索 -> 分析")
results = chain.execute({"topic": "AI Agent 市场规模"})

print("\n\n📊 工具链执行结果:")
for r in results:
    print(f"\n{r['step']} ({r['tool']}):")
    print(f"  结果: {json.dumps(r['result'], ensure_ascii=False)[:200]}...")

---

## 4. 工具选择策略

### 4.1 基于描述的工具选择

```python
class ToolSelector:
    """工具选择器"""
    
    def __init__(self, tool_registry: ToolRegistry):
        self.tool_registry = tool_registry
    
    def select_tools(self, query: str, top_k: int = 2) -> List[str]:
        """基于查询选择相关工具"""
        # 简单的关键词匹配
        tool_scores = {}
        
        for tool_name, tool in self.tool_registry.tools.items():
            score = self._calculate_relevance(query, tool)
            tool_scores[tool_name] = score
        
        # 返回得分最高的工具
        sorted_tools = sorted(tool_scores.items(), key=lambda x: x[1], reverse=True)
        return [name for name, score in sorted_tools[:top_k] if score > 0]
    
    def _calculate_relevance(self, query: str, tool: Tool) -> float:
        """计算查询与工具的相关性"""
        query_words = set(query.lower().split())
        desc_words = set(tool.description.lower().split())
        
        overlap = len(query_words & desc_words)
        return overlap / max(len(query_words), 1)

selector = ToolSelector(registry)

test_queries = [
    "北京天气怎么样?",
    "帮我搜索一下最新新闻",
    "计算一下这个公式"
]

for query in test_queries:
    tools = selector.select_tools(query)
    print(f"\n查询: {query}")
    print(f"推荐工具: {tools}")
```

---

## 5. 错误处理与重试机制

```python
class ToolExecutor:
    """带错误处理的工具执行器"""
    
    def __init__(self, tool_registry: ToolRegistry, max_retries: int = 3):
        self.tool_registry = tool_registry
        self.max_retries = max_retries
    
    def execute_with_retry(self, tool_name: str, **kwargs) -> Dict:
        """带重试的执行"""
        last_error = None
        
        for attempt in range(self.max_retries):
            try:
                result = self.tool_registry.execute(tool_name, **kwargs)
                return {
                    "success": True,
                    "result": result,
                    "attempts": attempt + 1
                }
            except Exception as e:
                last_error = str(e)
                print(f"   尝试 {attempt + 1} 失败: {last_error}")
                if attempt < self.max_retries - 1:
                    print(f"   等待重试...")
        
        return {
            "success": False,
            "error": last_error,
            "attempts": self.max_retries
        }
```

---

## 6. 小结

### 核心要点

1. **Function Calling** 让 LLM 能够调用外部工具扩展能力
2. **工具定义** 需要清晰的名称、描述和参数 Schema
3. **工具注册中心** 管理所有可用工具
4. **工具链** 支持多个工具的串行执行
5. **错误处理** 重试机制和异常处理很重要

### 最佳实践

| 实践 | 说明 |
|------|------|
| 清晰的工具描述 | 帮助 LLM 正确选择工具 |
| 参数验证 | 在执行前验证参数类型和范围 |
| 超时控制 | 防止工具执行时间过长 |
| 结果格式化 | 统一工具返回格式 |
| 错误处理 | 优雅处理工具执行失败 |

### 下一步

- [../04_projects/01_research_assistant.ipynb](../04_projects/01_research_assistant.ipynb) - 研究助手项目
- 探索更多实际工具集成（数据库、API、文件系统等）

---

## 参考资源

- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [LangChain Tools](https://python.langchain.com/docs/modules/agents/tools/)
- [Function Calling Best Practices](https://platform.openai.com/docs/guides/function-calling/best-practices)